# Setup

In [1]:
import numpy as np
from pathlib import Path
import pandas as pd
import torch
import random
from utils.utils import *
from data.simulate_walk_the_book import *
from utils.datastuff import TrainCfg
from utils.train import train_val
from utils.test import generate_test_loader, generate_test_predictions
from data.simulate_walk_the_book import simulate_walk_the_book
import warnings


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Fix randomness for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

warnings.filterwarnings("ignore")

device: cpu


In [2]:

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [3]:
# are we consuming asks or bids?
ASK_PRICE_COLS = ['ask_price_1', 'ask_price_2', 'ask_price_3', 'ask_price_4', 'ask_price_5']
ASK_VOL_COLS = ['ask_vol_1', 'ask_vol_2', 'ask_vol_3', 'ask_vol_4', 'ask_vol_5']
BID_PRICE_COLS = ['bid_price_1', 'bid_price_2', 'bid_price_3', 'bid_price_4', 'bid_price_5']
BID_VOL_COLS = ['bid_vol_1', 'bid_vol_2', 'bid_vol_3', 'bid_vol_4', 'bid_vol_5']
sides = ["bid", "ask"]

In [4]:
import sys, os

# Paths and volume_to_fill
# root_path = DATA_PATH
root_path = "../data"
root = Path(root_path) if Path(root_path).exists() else Path.cwd()

holdout_csv = pd.read_csv("holdout_split.csv")
holdout_csv["anonymized_id"] = holdout_csv["anonymized_id"].astype("uint64")

holdout_ids = set(
    holdout_csv.loc[holdout_csv["split"] == "holdout", "anonymized_id"]
)
dev_ids = set(
    holdout_csv.loc[holdout_csv["split"] == "dev", "anonymized_id"]
)

import sys
if str(root / "src") not in sys.path:
    sys.path.append(str(root / "src"))

data_assets = [folder_name for folder_name in os.listdir(root) if "USDT" in folder_name]

X_dfs = {}
y_dfs = {}
assets_ids = {}

X_dfs_ho = {}
y_dfs_ho = {}
assets_ids_ho = {}

for data_asset in data_assets:

    symbol_dir = root / data_asset
    X_path = symbol_dir / "X_train.parquet"
    Y_path = symbol_dir / "y_train.parquet"
    X_test_path = symbol_dir / "X_test.parquet"
    vol_path = symbol_dir / "vol_to_fill.txt"

    volume_to_fill = None
    if vol_path.exists():
        import re
        with open(vol_path) as f:
            m = re.search(r"([\d.]+)", f.read())
        if m:
            volume_to_fill = float(m.group(1))

    X_df = pd.read_parquet(X_path)
    y_df = pd.read_parquet(Y_path)

    ids = pd.Series(X_df["anonymized_id"].unique(), name="anonymized_id")

    last_time = X_df["time_in_hour"].sort_values().iloc[-1]
    future_times = pd.Series(
        [last_time + pd.Timedelta(seconds=i) for i in range(1, 61)],
        name="time_in_hour"
    )

    past_times = pd.Series(
        X_df["time_in_hour"].sort_values().unique(),
        name="time_in_hour"
    )

    full_grid = (
        ids.to_frame()
        .merge(future_times.to_frame(), how="cross")
    )

    full_grid_X = (
        ids.to_frame()
        .merge(past_times.to_frame(), how="cross")
    )

    y_df = full_grid.merge(
        y_df,
        on=["anonymized_id", "time_in_hour"],
        how="left"
    )

    X_df = full_grid_X.merge(
        X_df,
        on=["anonymized_id", "time_in_hour"],
        how="left"
    )

    # price imputation
    price_cols = ASK_PRICE_COLS + BID_PRICE_COLS + ["close"]
    X_df[price_cols] = X_df.groupby("anonymized_id")[price_cols].transform(lambda x: x.ffill().bfill())
    y_df[price_cols] = y_df.groupby("anonymized_id")[price_cols].transform(lambda x: x.ffill().bfill())

    # volume imputation
    volume_cols = BID_VOL_COLS + ASK_VOL_COLS + ["volume"]
    X_df[volume_cols] = X_df[volume_cols].fillna(0)
    y_df[volume_cols] = y_df[volume_cols].fillna(0)

    X_df["anonymized_id"] = X_df["anonymized_id"].astype("uint64")
    y_df["anonymized_id"] = y_df["anonymized_id"].astype("uint64")

    X_dev = X_df[X_df["anonymized_id"].isin(dev_ids)]
    y_dev = y_df[y_df["anonymized_id"].isin(dev_ids)]

    X_ho = X_df[X_df["anonymized_id"].isin(holdout_ids)]
    y_ho = y_df[y_df["anonymized_id"].isin(holdout_ids)]

    X_dfs[data_asset] = X_dev
    y_dfs[data_asset] = y_dev
    assets_ids[data_asset] = pd.Series(X_dev["anonymized_id"].unique())

    X_dfs_ho[data_asset] = X_ho
    y_dfs_ho[data_asset] = y_ho
    assets_ids_ho[data_asset] = pd.Series(X_ho["anonymized_id"].unique())


In [5]:
volumes_to_fill = {}

for asset in data_assets:
    vol_path = root / asset / "vol_to_fill.txt"
    if vol_path.exists():
        with open(vol_path) as f:
            m = re.search(r"([\d.]+)", f.read())
        if m:            
            volumes_to_fill[asset] = float(m.group(1)) 

import matplotlib.pyplot as plt

def plot_volumes_to_fill():
    # Sort dictionary by values
    sorted_items = sorted(volumes_to_fill.items(), key=lambda x: x[1])

    # Unzip keys and values
    labels, values = zip(*sorted_items)

    # Create bar plot
    plt.figure()
    plt.bar(labels, values)

    # Optional labels
    plt.xlabel("Items")
    plt.ylabel("Values")
    plt.title("Volumes to Fill by Asset")

    plt.show()

In [6]:
# def find_optimal_k(ids, y_df, side_weight, volume_to_fill):
#     best_k = None
#     best_bps = np.inf

#     for k in range(1, 61):
#         bpss = []
#         for anon_id in ids:

#             df_inst = y_df[y_df["anonymized_id"] == anon_id].sort_values("time_in_hour")

#             ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
#             ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
#             bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
#             bid_vols = df_inst[BID_VOL_COLS].to_numpy()
#             close_price = df_inst['close'].dropna().iloc[-1]

#             weights = np.array([0.]* (60-k) + [1.]* k)
#             weights /= weights.sum()

#             if not np.isclose(weights.sum(), 1):
#                 print(f"{anon_id} doesn't have weights that sum to 1.")

#             id_positions = weights * volume_to_fill

#             id_positions *= side_weight

#             model_vol, model_avg_price = simulate_walk_the_book(
#                 id_positions, ask_prices, ask_vols, bid_prices, bid_vols
#             )
            
#             if model_vol > 0 and not np.isnan(model_avg_price):
#                 rel_error = np.abs(model_avg_price - close_price) / close_price
#                 vol_penalty = min(100.0, volume_to_fill / model_vol)
#                 bpss.append(rel_error * vol_penalty * 10000)
        
#         if np.mean(bpss) < best_bps:
#             best_bps = np.mean(bpss)
#             best_k = k

#     return best_k, best_bps

# rows = []

# for side in sides:
#     side_weight = 1 if side == "ask" else -1
    
#     for data_asset in data_assets:
#         y_df = y_dfs[data_asset]
#         ids = assets_ids[data_asset]
#         volume_to_fill = volumes_to_fill[data_asset]

#         optimal_k, optimal_bps = find_optimal_k(ids, y_df, side_weight, volume_to_fill)

#         rows.append({
#             "side": side,
#             "Asset": data_asset,
#             "Optimal_K": optimal_k,
#             "bps_optimal_K": optimal_bps
#         })

# # build DataFrame
# df_k = pd.DataFrame(rows)

# # set MultiIndex
# df_k = df_k.set_index(["side", "Asset"]).sort_index()

# df_k.to_csv("optimal_k_dev.csv")

# df_k

In [7]:
df_k = pd.read_csv("optimal_k_dev.csv", index_col=[0, 1])

# Ultimate Benchmark of Above Approaches

we want a MultiIndex Asset - Model \in {optimal K, capping(funcs), redistribute(funcs), redistribute(decreasing caps), redistribute(decreasing caps with slope)}

features are [bps, % filled]

In [8]:
benchmark = pd.DataFrame(
    columns=["bps", "percent_filled"]
)

benchmark.index = pd.MultiIndex.from_tuples(
    [],
    names=["Side", "Asset", "Volume to Fill", "strategy"]
)

benchmark

,,,,bps,percent_filled
Side,Asset,Volume to Fill,strategy,,


In [9]:
positions_cols = [f"weight_{i}" for i in range(60)]
strategy = "optimal_k"

for side in sides:

    side_weight = 1 if side == "ask" else -1

    for data_asset in data_assets:

        X_df = X_dfs_ho[data_asset]
        y_df = y_dfs_ho[data_asset]
        ids = assets_ids_ho[data_asset]
        optimal_k = int(df_k.loc[side, data_asset]["Optimal_K"])
        volume_to_fill = volumes_to_fill[data_asset]
        num_ids = len(ids)

        bpss = []
        percents_filled = []

        benchmark.loc[(side, data_asset, volume_to_fill, strategy), positions_cols] = 0

        for anon_id in ids:

            df_inst = y_df[y_df["anonymized_id"] == anon_id].sort_values("time_in_hour")

            ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
            ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
            bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
            bid_vols = df_inst[BID_VOL_COLS].to_numpy()
            close_price = df_inst['close'].dropna().iloc[-1]

            weights = np.array([0.]* (60-optimal_k) + [1.]* optimal_k)
            weights /= weights.sum()

            if not np.isclose(weights.sum(), 1):
                print(f"{anon_id} doesn't have weights that sum to 1.")

            id_positions = weights * volume_to_fill

            id_positions *= side_weight

            model_vol, model_avg_price = simulate_walk_the_book(
                id_positions, ask_prices, ask_vols, bid_prices, bid_vols
            )
            if model_vol > 0 and not np.isnan(model_avg_price):
                rel_error = np.abs(model_avg_price - close_price) / close_price
                vol_penalty = min(100.0, volume_to_fill / model_vol)
                bpss.append(rel_error * vol_penalty * 10000)
                percents_filled.append(model_vol / volume_to_fill * 100)

        benchmark.loc[(side, data_asset, volume_to_fill, strategy), "bps"] = np.mean(bpss)
        benchmark.loc[(side, data_asset, volume_to_fill, strategy), "percent_filled"] = np.round(np.mean(percents_filled), 1)
        benchmark.loc[(side, data_asset, volume_to_fill, strategy), positions_cols] += id_positions / num_ids


KeyboardInterrupt: 

In [ ]:
benchmark

## Cap the Allocated Volume Based on Previous Volume

capping analysis:
- plotting function for objective components
- plotting pipeline for redistribution & capping with linear, **2, **3, **4, **5

In [ ]:
methods = ["capped", "distributed"]
functions = ["linear", "**2", "**3", "**5", "**10"]
percentiles = ["mean", 50, 80]


In [ ]:
cap_col = f"{side}_vol_1"

for side in sides:
    side_weight = 1 if side == "ask" else -1
    for data_asset in data_assets:

        X_df = X_dfs_ho[data_asset]
        y_df = y_dfs_ho[data_asset]
        ids = assets_ids_ho[data_asset]
        volume_to_fill = volumes_to_fill[data_asset]

        for percentile in percentiles:
            for func in functions:

                bpss = []
                percents_filled = []

                benchmark.loc[(side, data_asset, volume_to_fill, strategy), positions_cols] = 0

                for anon_id in ids:

                    df_inst = y_df[y_df["anonymized_id"] == anon_id].sort_values("time_in_hour")
                    X_df_inst = X_df[X_df["anonymized_id"] == anon_id].sort_values("time_in_hour")

                    if percentile == "mean":
                        volume_cap = np.mean(X_df_inst[cap_col])
                    else:
                        volume_cap = np.percentile(X_df_inst[cap_col], percentile)

                    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
                    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
                    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
                    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
                    close_price = df_inst['close'].dropna().iloc[-1]
                    
                    if volume_cap == 0:
                        print("mean used!!!")
                        volume_cap = np.mean(X_df_inst[cap_col])

                    weights = np.linspace(0, 1, 60)

                    if func != "linear":
                        weights = weights ** int(func[2:])

                    weights /= weights.sum()

                    if not np.isclose(weights.sum(), 1):
                        print(f"{anon_id} doesn't have weights that sum to 1.")

                    # simply cap
                    id_positions = np.minimum(weights * volume_to_fill, volume_cap)

                    id_positions *= side_weight

                    model_vol, model_avg_price = simulate_walk_the_book(
                        id_positions, ask_prices, ask_vols, bid_prices, bid_vols
                    )
                    
                    if model_vol > 0 and not np.isnan(model_avg_price):
                        rel_error = np.abs(model_avg_price - close_price) / close_price
                        vol_penalty = min(100.0, volume_to_fill / model_vol)

                        bpss.append(rel_error * vol_penalty * 10000)
                        percents_filled.append(model_vol / volume_to_fill * 100)

                strategy = f"capped_{func}_{percentile}"
                benchmark.loc[(side, data_asset, volume_to_fill, strategy), "bps"] = np.mean(bpss)
                benchmark.loc[(side, data_asset, volume_to_fill, strategy), "percent_filled"] = np.round(np.mean(percents_filled), 1)
                benchmark.loc[(side, data_asset, volume_to_fill, strategy), positions_cols] = id_positions / num_ids



## Redistribute if Cap Exceeded

**cap**(**function**) + **redistribute**()

In [ ]:
def capped_allocation(weights, total_volume, cap):
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()

    allocation = weights * total_volume
    capped = np.minimum(allocation, cap)

    deficit = total_volume - capped.sum()

    while deficit > 1e-9:
        free = capped < cap
        if not np.any(free):
            break

        w_free = weights[free]
        w_free = w_free / w_free.sum()

        increment = np.zeros_like(capped)
        increment[free] = w_free * deficit

        new_vals = np.minimum(capped + increment, cap)
        delta = new_vals - capped

        capped = new_vals
        deficit -= delta.sum()

    return capped


cap_col = f"{side}_vol_1"

for side in sides:
    side_weight = 1 if side == "ask" else -1
    for data_asset in data_assets:

        X_df = X_dfs_ho[data_asset]
        y_df = y_dfs_ho[data_asset]
        ids = assets_ids_ho[data_asset]
        volume_to_fill = volumes_to_fill[data_asset]

        for percentile in percentiles:
            for func in functions:

                bpss = []
                percents_filled = []

                benchmark.loc[(side, data_asset, volume_to_fill, strategy), positions_cols] = 0

                for anon_id in ids:

                    df_inst = y_df[y_df["anonymized_id"] == anon_id].sort_values("time_in_hour")
                    X_df_inst = X_df[X_df["anonymized_id"] == anon_id].sort_values("time_in_hour")

                    if percentile == "mean":
                        volume_cap = np.mean(X_df_inst[cap_col])
                    else:
                        volume_cap = np.percentile(X_df_inst[cap_col], percentile)

                    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
                    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
                    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
                    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
                    close_price = df_inst['close'].dropna().iloc[-1]
                    
                    if volume_cap == 0:
                        print("mean used!!!")
                        volume_cap = np.mean(X_df_inst[cap_col])

                    weights = np.linspace(0, 1, 60)

                    if func != "linear":
                        weights = weights ** int(func[2:])

                    id_positions = capped_allocation(weights, volume_to_fill, volume_cap)

                    id_positions *= side_weight

                    model_vol, model_avg_price = simulate_walk_the_book(
                        id_positions, ask_prices, ask_vols, bid_prices, bid_vols
                    )
                    
                    if model_vol > 0 and not np.isnan(model_avg_price):
                        rel_error = np.abs(model_avg_price - close_price) / close_price
                        vol_penalty = min(100.0, volume_to_fill / model_vol)

                        bpss.append(rel_error * vol_penalty * 10000)
                        percents_filled.append(model_vol / volume_to_fill * 100)

                strategy = f"redistributed_{func}_{percentile}"
                benchmark.loc[(side, data_asset, volume_to_fill, strategy), "bps"] = np.mean(bpss)
                benchmark.loc[(side, data_asset, volume_to_fill, strategy), "percent_filled"] = np.round(np.mean(percents_filled), 1)
                benchmark.loc[(side, data_asset, volume_to_fill, strategy), positions_cols] = id_positions / num_ids

In [ ]:
benchmark_copy = benchmark.copy()

## Redistribute if Cap Exceeded but Decrease Caps based on Distance to Final Sec

**cap**(**function**) + **redistribute**(**decreasing caps**)

In [ ]:
second_caps_start = [20, 30, 40]

In [ ]:
for side in sides:
    side_weight = 1 if side == "ask" else -1
    for data_asset in data_assets:

        X_df = X_dfs_ho[data_asset]
        y_df = y_dfs_ho[data_asset]
        ids = assets_ids_ho[data_asset]
        volume_to_fill = volumes_to_fill[data_asset]

        for percentile in percentiles:
            for func in functions:
                for second_cap in second_caps_start:

                    bpss = []
                    percents_filled = []

                    benchmark.loc[(side, data_asset, volume_to_fill, strategy), positions_cols] = 0

                    for anon_id in ids:

                        df_inst = y_df[y_df["anonymized_id"] == anon_id].sort_values("time_in_hour")
                        X_df_inst = X_df[X_df["anonymized_id"] == anon_id].sort_values("time_in_hour")

                        if percentile == "mean":
                            volume_cap = np.mean(X_df_inst[cap_col])
                        else:
                            volume_cap = np.percentile(X_df_inst[cap_col], percentile)

                        if volume_cap == 0:
                            print("mean used!!!")
                            volume_cap = np.mean(X_df_inst[cap_col])
                        
                        volume_caps = np.append(np.array([0]*second_cap), np.linspace(0, volume_cap, 60-second_cap)) # linearly increasing caps from second_cap onwards

                        ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
                        ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
                        bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
                        bid_vols = df_inst[BID_VOL_COLS].to_numpy()
                        close_price = df_inst['close'].dropna().iloc[-1]

                        weights = np.linspace(0, 1, 60)

                        if func != "linear":
                            weights = weights ** int(func[2:])

                        id_positions = capped_allocation(weights, volume_to_fill, volume_caps)

                        id_positions *= side_weight

                        model_vol, model_avg_price = simulate_walk_the_book(
                            id_positions, ask_prices, ask_vols, bid_prices, bid_vols
                        )
                        
                        if model_vol > 0 and not np.isnan(model_avg_price):
                            rel_error = np.abs(model_avg_price - close_price) / close_price
                            vol_penalty = min(100.0, volume_to_fill / model_vol)

                            bpss.append(rel_error * vol_penalty * 10000)
                            percents_filled.append(model_vol / volume_to_fill * 100)

                    strategy = f"redistributed_second{second_cap}_{func}_{percentile}"
                    benchmark.loc[(side, data_asset, volume_to_fill, strategy), "bps"] = np.mean(bpss)
                    benchmark.loc[(side, data_asset, volume_to_fill, strategy), "percent_filled"] = np.round(np.mean(percents_filled), 1)
                    benchmark.loc[(side, data_asset, volume_to_fill, strategy), positions_cols] = id_positions / num_ids

## Redistribute if Cap Exceeded but Decrease Caps based on Distance to Final Sec

maybe we could also include this volatility marker in there somehow. if volatility std is high, we make the caps decrease slower. if std is low, we decrease quickly. this should be unique for each id.

so the final model is:

**cap**(**function**) + **redistribute**(**decreasing caps** where **slope = 1 / volatility std**) -> 5 components

note that function could be replaced with Maaz's adaptive K approach

This is the ultimate model lmao

In [ ]:
def vol_regime(x):
        low = x.quantile(0.2)
        high = x.quantile(0.8) 
        return pd.cut(
            x,
            bins=[-np.inf, low, high, np.inf],
            labels=["low", "normal", "high"]
        )

volatility_window = 120 # last 2 minutes volatility

for i in X_dfs_ho:
    X_df = X_dfs_ho[i]

    X_df["midprice"] = (X_df["ask_price_1"] + X_df["bid_price_1"]) / 2

    # volatility measure
    X_df["midvolatility"] = X_df.groupby("anonymized_id")["midprice"].transform(lambda x: np.log(x).diff()).fillna(0)

    X_df["midvolatilitystd"] = X_df.groupby("anonymized_id")["midvolatility"].transform(lambda x: x.rolling(volatility_window).std()) 

    X_df["vol_regime"] = X_df.groupby("anonymized_id")["midvolatilitystd"].transform(vol_regime)

In [ ]:
last_sec = X_df["time_in_hour"].sort_values().iloc[-1]

In [ ]:
benchmark = benchmark_copy.copy()

In [ ]:
for side in sides:
    side_weight = 1 if side == "ask" else -1
    for data_asset in data_assets:

        X_df = X_dfs_ho[data_asset]
        y_df = y_dfs_ho[data_asset]
        ids = assets_ids_ho[data_asset]
        volume_to_fill = volumes_to_fill[data_asset]

        for percentile in percentiles:
            for func in functions:

                bpss = []
                percents_filled = []

                benchmark.loc[(side, data_asset, volume_to_fill, strategy), positions_cols] = 0

                for anon_id in ids:

                    df_inst = y_df[y_df["anonymized_id"] == anon_id].sort_values("time_in_hour")
                    X_df_inst = X_df[X_df["anonymized_id"] == anon_id].sort_values("time_in_hour")

                    ###
                    vol_regime = X_df[X_df["anonymized_id"] == anon_id][X_df["time_in_hour"] == last_sec]["vol_regime"].iloc[0]

                    if vol_regime == "low":
                        volatility_cap = 45
                    elif vol_regime == "high":
                        volatility_cap = 15
                    elif vol_regime == "normal":
                        volatility_cap = 30
                    ###

                    if percentile == "mean":
                        volume_cap = np.mean(X_df_inst[cap_col])
                    else:
                        volume_cap = np.percentile(X_df_inst[cap_col], percentile)

                    if volume_cap == 0:
                        print("mean used!!!")
                        volume_cap = np.mean(X_df_inst[cap_col])
                    
                    volume_caps = np.append(np.array([0]*volatility_cap), np.linspace(0, volume_cap, 60-volatility_cap)) # linearly increasing caps from second_cap onwards

                    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
                    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
                    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
                    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
                    close_price = df_inst['close'].dropna().iloc[-1]
                    

                    weights = np.linspace(0, 1, 60)

                    if func != "linear":
                        weights = weights ** int(func[2:])

                    id_positions = capped_allocation(weights, volume_to_fill, volume_caps)

                    id_positions *= side_weight

                    model_vol, model_avg_price = simulate_walk_the_book(
                        id_positions, ask_prices, ask_vols, bid_prices, bid_vols
                    )
                    
                    if model_vol > 0 and not np.isnan(model_avg_price):
                        rel_error = np.abs(model_avg_price - close_price) / close_price
                        vol_penalty = min(100.0, volume_to_fill / model_vol)

                        bpss.append(rel_error * vol_penalty * 10000)
                        percents_filled.append(model_vol / volume_to_fill * 100)

                strategy = f"redistributed_volcap_{func}_{percentile}"
                benchmark.loc[(side, data_asset, volume_to_fill, strategy), "bps"] = np.mean(bpss)
                benchmark.loc[(side, data_asset, volume_to_fill, strategy), "percent_filled"] = np.round(np.mean(percents_filled), 1)
                benchmark.loc[(side, data_asset, volume_to_fill, strategy), positions_cols] = id_positions / num_ids

# Ideal Strategy Analysis

## Best Approaches

In [ ]:
def get_top_with_optimal(benchmark, asset, top_n=3):

    df = benchmark.xs(asset, level="Asset", drop_level=False)

    df_no_opt = df[df.index.get_level_values("strategy") != "optimal_k"]
    top = df_no_opt.sort_values("bps").head(top_n)

    optimal = df[df.index.get_level_values("strategy") == "optimal_k"]

    result = pd.concat([top, optimal])

    result = result.sort_values("bps")

    return result

In [ ]:
get_top_with_optimal(benchmark.loc["ask"], data_assets[0], top_n=1).drop(columns=positions_cols)

In [ ]:
get_top_with_optimal(benchmark.loc["bid"], data_assets[0], top_n=1).drop(columns=positions_cols)

In [ ]:
get_top_with_optimal(benchmark.loc["ask"], data_assets[1], top_n=1).drop(columns=positions_cols)

In [ ]:
get_top_with_optimal(benchmark.loc["bid"], data_assets[1], top_n=1).drop(columns=positions_cols)

In [ ]:
get_top_with_optimal(benchmark.loc["ask"], data_assets[2], top_n=1).drop(columns=positions_cols)

In [ ]:
get_top_with_optimal(benchmark.loc["bid"], data_assets[2], top_n=1).drop(columns=positions_cols)

In [ ]:
get_top_with_optimal(benchmark.loc["ask"], data_assets[3], top_n=1).drop(columns=positions_cols)

In [ ]:
get_top_with_optimal(benchmark.loc["bid"], data_assets[3], top_n=1).drop(columns=positions_cols)

In [ ]:
get_top_with_optimal(benchmark.loc["ask"], data_assets[4], top_n=1).drop(columns=positions_cols)

In [ ]:
get_top_with_optimal(benchmark.loc["bid"], data_assets[4], top_n=1).drop(columns=positions_cols)

In [ ]:
get_top_with_optimal(benchmark.loc["ask"], data_assets[5], top_n=1).drop(columns=positions_cols)

In [ ]:
get_top_with_optimal(benchmark.loc["bid"], data_assets[5], top_n=1).drop(columns=positions_cols)

In [ ]:
get_top_with_optimal(benchmark.loc["ask"], data_assets[6], top_n=1).drop(columns=positions_cols)

In [ ]:
get_top_with_optimal(benchmark.loc["bid"], data_assets[6], top_n=1).drop(columns=positions_cols)

In [ ]:
benchmark.to_csv(f"benchmark_results.csv")

# EDA of Benchmark

In [19]:
old_benchmark = pd.read_csv("benchmark_results.csv", index_col=[0, 1, 2, 3])

In [20]:
old_benchmark#.index

bps  \
Side Asset   Volume to Fill strategy                                   
bid  BTCUSDT 4.0            optimal_k                       1.292175   
     XRPUSDT 13736.0        optimal_k                       4.198014   
     ETHUSDT 55.0           optimal_k                       3.068327   
     ADAUSDT 10436.0        optimal_k                       6.927122   
     SOLUSDT 315.0          optimal_k                       5.585245   
...                                                              ...   
ask  LTCUSDT 51.0           redistributed_volcap_**10_50    5.213541   
                            redistributed_volcap_linear_80  5.157573   
                            redistributed_volcap_**2_80     5.170205   
                            redistributed_volcap_**3_80     5.163190   
                            redistributed_volcap_**5_80     5.147485   

                                                            percent_filled  \
Side Asset   Volume to Fill strategy                                         
bid  BTCUSDT 4.0            optimal_k                                 99.8   
     XRPUSDT 13736.0        optimal_k                                100.0   
     ETHUSDT 55.0           optimal_k                                100.0   
     ADAUSDT 10436.0        optimal_k                                 99.5   
     SOLUSDT 315.0          optimal_k                                 99.9   
...                                                                    ...   
ask  LTCUSDT 51.0           redistributed_volcap_**10_50              98.8   
                            redistributed_volcap_linear_80            99.7   
                            redistributed_volcap_**2_80               99.6   
                            redistributed_volcap_**3_80               99.5   
                            redistributed_volcap_**5_80               99.3   

                                                            weight_0  \
Side Asset   Volume to Fill strategy                                   
bid  BTCUSDT 4.0            optimal_k                            0.0   
     XRPUSDT 13736.0        optimal_k                            0.0   
     ETHUSDT 55.0           optimal_k                            0.0   
     ADAUSDT 10436.0        optimal_k                            0.0   
     SOLUSDT 315.0          optimal_k                            0.0   
...                                                              ...   
ask  LTCUSDT 51.0           redistributed_volcap_**10_50         0.0   
                            redistributed_volcap_linear_80       0.0   
                            redistributed_volcap_**2_80          0.0   
                            redistributed_volcap_**3_80          0.0   
                            redistributed_volcap_**5_80          0.0   

                                                            weight_1  \
Side Asset   Volume to Fill strategy                                   
bid  BTCUSDT 4.0            optimal_k                            0.0   
     XRPUSDT 13736.0        optimal_k                            0.0   
     ETHUSDT 55.0           optimal_k                            0.0   
     ADAUSDT 10436.0        optimal_k                            0.0   
     SOLUSDT 315.0          optimal_k                            0.0   
...                                                              ...   
ask  LTCUSDT 51.0           redistributed_volcap_**10_50         0.0   
                            redistributed_volcap_linear_80       0.0   
                            redistributed_volcap_**2_80          0.0   
                            redistributed_volcap_**3_80          0.0   
                            redistributed_volcap_**5_80          0.0   

                                                            weight_2  \
Side Asset   Volume to Fill strategy                                   
bid  BTCUSDT 4.0            optimal_k                            0.0   
     XRPUSDT 13736.0   

In [14]:
df_k

Optimal_K  bps_optimal_K
side Asset                             
ask  ADAUSDT          17       5.439439
     BTCUSDT          20       1.223731
     DOGEUSDT         34       5.087866
     ETHUSDT          30       2.588819
     LTCUSDT          28       5.270704
     SOLUSDT          20       5.229563
     XRPUSDT          20       3.554792
bid  ADAUSDT          10       7.603906
     BTCUSDT          14       1.171398
     DOGEUSDT         21       5.388471
     ETHUSDT          31       2.742268
     LTCUSDT          10       7.688484
     SOLUSDT          10       5.600838
     XRPUSDT          20       4.297997

In [15]:
old_benchmark.xs('optimal_k', level='strategy', axis=0).drop(columns=positions_cols)

KeyError: 'Level strategy not found'